In [33]:
import pandas as pd
import numpy as np
import scipy.stats as stats
from scipy.stats import f,t,ttest_1samp,ttest_ind,f_oneway,norm
from statsmodels.stats.proportion import proportion_confint
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv('/Users/Marcy_Student/Desktop/marcy/DA2025_Lectures/Mod4_StatsReview/exams.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


In [ ]:
df.columns = (
            df.columns.str.lower()
            .str.replace(r"[^a-z0-9]+","_", regex = True)
            .str.strip("_")
)
# creating an overall score than add all the scores together
df['overall_score'] = df[['math_score','reading_score','writing_score']].mean(axis=1)

# creating a passed variable, which says if somebody has passed or no
# I set the threshold to 70
PASS_CUT = 70
df['passed'] = df['overall_score']> PASS_CUT

#lowercase the completion level with the test preparation variable
df['completed'] = (df['test_preparation_course'].str.lower() == 'completed')
df

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score,overall_score,passed,completed
0,male,group A,high school,standard,completed,67,67,63,65.666667,False,True
1,female,group D,some high school,free/reduced,none,40,59,55,51.333333,False,False
2,male,group E,some college,free/reduced,none,59,60,50,56.333333,False,False
3,male,group B,high school,standard,none,77,78,68,74.333333,True,False
4,male,group E,associate's degree,standard,completed,78,73,68,73.000000,True,True
...,...,...,...,...,...,...,...,...,...,...,...
995,male,group C,high school,standard,none,73,70,65,69.333333,False,False
996,male,group D,associate's degree,free/reduced,completed,85,91,92,89.333333,True,True
997,female,group C,some high school,free/reduced,none,32,35,41,36.000000,False,False
998,female,group C,some college,standard,none,73,74,82,76.333333,True,False


### Probability Rules you'll actually use 
Complement: if 72% pass, then 28% don’t (you’ll use this constantly when interpreting metrics).

Addition/Multiplication: when to add vs multiply; independence warning.

**From data: compute P(pass), P(prep), P(pass ∩ prep), P(pass | prep).**

In [25]:
# function to obtain the probability
def P(s):  return s.mean()
# compute the probabilities 
P_pass = P(df['passed'])
P_prep = P(df['completed'])
P_pass_and_prep = P(df['passed'] & df['completed'])
P_pass_given_prep = P_pass_and_prep / P_prep
P_not_pass = 1 - P_pass 

print (f'P(pass): {P_pass:.3f}')
print (f'P(prep): {P_prep:.3f}')
print (f'P(pass ∩ prep): {P_pass_and_prep:.3f}')
print (f"P(pass|prep): {P_pass_given_prep:.3f}")

P(pass): 0.439
P(prep): 0.335
P(pass ∩ prep): 0.194
P(pass|prep): 0.579


### Bayes' theorem = updated belief after evidence 

Your prior = baseline pass rate; evidence = who completed prep; posterior = P(pass | prep).

Compare both the (direct conditional from data) and the Bayes calculation below (prior, sensitivity, false-positive) -- should get the same answer! 


In [ ]:
prior = P_pass
tpr = P(df.loc[df["passed"], "completed"])   # P(prep|pass)
fpr = P(df.loc[~df['passed'],"completed"])  # P(prep|not pass)
def bayes_posterior(prior, p_B_given_A, p_B_given_notA):
    return (p_B_given_A*prior) / (p_B_given_A*prior + p_B_given_notA*(1-prior))
posterior_pass_given_prep = bayes_posterior(prior, tpr, fpr)

print(f"Bayes posterior P(pass | prep) = {posterior_pass_given_prep:.3f}")
#Check against the answer you got in the previous cell 

Bayes posterior P(pass | prep) = 0.579


### PDF vs CDF: how to read AND use them with data 

PDF (hist density) tells you the shape and “where values live.”

CDF tells you P(X ≤ x)—the probability up to a threshold (e.g., “% of students scoring ≤ 80”).

Subtract the CDF from 1 to get the probability of a student having a score greater than 80

In [41]:
x = df['math_score'].astype(float)
TARGET = 80
BINS = 15
hist_vals, bin_edges = np.histogram(x, bins=BINS, density=True)
#Get the probability of a math score of 80
cdf = (x <= TARGET).mean()
cdf


np.float64(0.812)

### Welch's t-test: Most common two-sample comparison in the real world

Question: “Did prep help math scores?” → compare prep_completed vs not.

Why Welch: you rarely know/assume equal variances in the "wild"

In [45]:
alpha = 0.05
a = df.loc[df['completed'],'math_score'].astype(float)
b = df.loc[~df['completed'],'math_score'].astype(float)
tt = stats.ttest_ind(a,b, equal_var=False)
print (f't={tt.statistic:.3f}, p={tt.pvalue:.3f}, reject H0 at {alpha} ? {tt.pvalue < alpha}')
print(f"Group means: completed={a.mean():.1f}, none={b.mean():.1f}")

t=4.851, p=0.000, reject H0 at 0.05 ? True
Group means: completed=69.7, none=64.7
